# 📊 Foundations of Statistics for Data & AI Work

> **Target outcome:** Stop treating statistics as magic — start seeing it as structured common sense.

Statistics is the science of extracting insight from data. It organises, analyses, and interprets information to uncover patterns and make decisions under **uncertainty**.

In machine learning and AI:
- ML models are opinions expressed in math — and those opinions are built on statistical foundations
- Statistics is how we measure uncertainty and risk
- Most AI failures are statistical misunderstandings, not model failures
- We use statistics to understand data distributions, identify useful features, validate model results, and make confident decisions

---

## 🗂️ What We'll Cover

| # | Topic |
|---|-------|
| 1 | Descriptive Statistics: Mean, Median, Mode, Variance, Std Dev |
| 2 | Probability Distributions (Normal & Binomial) |
| 3 | Hypothesis Testing |
| 4 | Confidence Intervals |
| 5 | Correlation vs. Causation |
| 6 | Introduction to Regression |
| 7 | Interactive Session: Applying Stats to the Diabetes Dataset |

---

**Dataset used throughout:** The [Pima Indians Diabetes Dataset](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) — 768 patient records with 8 clinical features and a binary outcome (diabetic or not). This dataset lets us apply every concept to a real-world medical problem.

In [ ]:
# ============================================================
# SETUP: Import all libraries we'll use in this notebook
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'
TEAL   = '#028090'
ORANGE = '#F4A261'
CORAL  = '#E76F51'
NAVY   = '#264653'

# Load the dataset
df = pd.read_csv('diabetes.csv')
print('Dataset shape:', df.shape)
df.head()

---
## 1. Descriptive Statistics: Making Sense of Your Data

Before training any model, we always start with **Exploratory Data Analysis (EDA)** — a process that relies on descriptive statistics to summarise the key characteristics of the data.

These summaries tell us:
- Where the data is **centred** (central tendency)
- How **spread out** the data is (variability)
- Whether there are **outliers** or data quality issues
- What **preprocessing** we might need

---

### 1.1 Measures of Central Tendency

#### 🎯 Mean — *The Sensitive Average*

The **mean** is the arithmetic average: add all values and divide by the count. It's the *balance point* of the data — every single value has a say.

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

- **Use when:** data is roughly symmetric (no extreme outliers)
- **In ML:** used to normalise features (feature scaling), loss functions like MSE
- **Weakness:** one extreme outlier can pull the mean far from where most data sits

#### ⚖️ Median — *The Stubborn Middle*

The **median** is the literal middle value when data is sorted. It doesn't care about the *size* of extremes — only their *position*.

- **Use when:** data is skewed or has extreme outliers
- **In ML:** used for imputing (filling in) missing values in skewed features
- **Strength:** robust to outliers — one billionaire in a room doesn't shift the median income much

#### 🏆 Mode — *The Crowd Favourite*

The **mode** is the most frequently occurring value.

- **Use when:** dealing with categorical data where arithmetic doesn't apply
- **In ML:** used in ensemble methods (majority voting), imputing categorical missing values
- **Example:** the most common browser used by website visitors

In [ ]:
# ============================================================
# MEAN, MEDIAN, MODE — on the Glucose feature
# ============================================================
glucose = df['Glucose']

mean_g   = glucose.mean()
median_g = glucose.median()
mode_g   = glucose.mode()[0]

print('=== Glucose (mg/dL) Central Tendency ===')
print(f'  Mean   : {mean_g:.2f}')
print(f'  Median : {median_g:.2f}')
print(f'  Mode   : {mode_g:.2f}')

# --- Visualise ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution with lines
axes[0].hist(glucose, bins=30, color=TEAL, alpha=0.7, edgecolor='white')
axes[0].axvline(mean_g,   color=CORAL,  lw=2.5, linestyle='--', label=f'Mean   = {mean_g:.1f}')
axes[0].axvline(median_g, color=ORANGE, lw=2.5, linestyle='-',  label=f'Median = {median_g:.1f}')
axes[0].axvline(mode_g,   color=NAVY,   lw=2.5, linestyle=':',  label=f'Mode   = {mode_g:.1f}')
axes[0].set_title('Glucose Distribution with Central Tendency Markers', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Glucose (mg/dL)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Right: comparison bar chart across all numeric features
features = ['Glucose', 'BloodPressure', 'BMI', 'Age']
means   = [df[f].mean()   for f in features]
medians = [df[f].median() for f in features]
x = np.arange(len(features))
width = 0.35
axes[1].bar(x - width/2, means,   width, label='Mean',   color=CORAL,  alpha=0.85)
axes[1].bar(x + width/2, medians, width, label='Median', color=ORANGE, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(features)
axes[1].set_title('Mean vs Median Across Features\n(close values → symmetric distribution)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Value')
axes[1].legend()

plt.tight_layout()
plt.show()

print()
print('💡 Notice: for Glucose, mean ≈ median → the distribution is fairly symmetric.')
print('   If mean >> median, the distribution is right-skewed (pulled up by high values).')

---
### 1.2 Measures of Variability — *The Nervous Energy*

The mean tells us *where* data sits. Variance and standard deviation tell us *how wild* the data is around that centre.

#### 📐 Variance — *The Raw Math Engine*

Variance is the **average of the squared distances** from the mean:

$$\sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2$$

**Why do we square?** If we just averaged the distances, positive and negative deviations would cancel to zero. Squaring makes everything positive and *punishes outliers* — big gaps look even bigger.

**The catch:** because we squared everything, the units are weird (e.g., "squared kg"). Hard to interpret directly.

#### 📏 Standard Deviation — *The Back-to-Reality Ruler*

Standard deviation is simply the **square root of variance**, bringing units back to reality:

$$\sigma = \sqrt{\sigma^2}$$

| SD Level | Meaning |
|----------|---------|
| **Low SD** | Data is calm — points cluster tightly around the mean |
| **High SD** | Data is frantic — points are scattered everywhere (high risk/unpredictability) |

#### 🔔 The 3-Sigma Rule (for Normal Distributions)

For normally distributed data:
- **68%** of values fall within ±1σ of the mean
- **95%** of values fall within ±2σ
- **99.7%** of values fall within ±3σ

Any value beyond ±3σ is considered a statistical outlier.

In [ ]:
# ============================================================
# VARIANCE & STANDARD DEVIATION — effect of outliers
# ============================================================

# Use BMI column from diabetes dataset
# Also simulate a "clean" version by removing top outliers
bmi = df['BMI'].replace(0, np.nan).dropna()
bmi_no_outliers = bmi[bmi < bmi.quantile(0.99)]

print('=== BMI Statistics ===')
print(f'Full data  — Mean: {bmi.mean():.2f}, Std: {bmi.std():.2f}, Variance: {bmi.var():.2f}')
print(f'No outliers— Mean: {bmi_no_outliers.mean():.2f}, Std: {bmi_no_outliers.std():.2f}, Variance: {bmi_no_outliers.var():.2f}')

# --- Full descriptive summary of the dataset ---
print('\n=== Full Dataset — Descriptive Statistics ===')
print(df.describe().round(2))

# --- Visualise: 3-sigma rule on Glucose ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: 3-sigma zones on Glucose
mu  = glucose.mean()
sig = glucose.std()
x_range = np.linspace(glucose.min() - 10, glucose.max() + 10, 300)
pdf = stats.norm.pdf(x_range, mu, sig)

axes[0].plot(x_range, pdf, color=TEAL, lw=2.5)
axes[0].fill_between(x_range, pdf,
    where=(x_range >= mu - sig) & (x_range <= mu + sig),
    alpha=0.5, color=TEAL, label='±1σ  (68%)')
axes[0].fill_between(x_range, pdf,
    where=((x_range >= mu - 2*sig) & (x_range < mu - sig)) |
          ((x_range > mu + sig) & (x_range <= mu + 2*sig)),
    alpha=0.35, color=ORANGE, label='±2σ  (95%)')
axes[0].fill_between(x_range, pdf,
    where=((x_range >= mu - 3*sig) & (x_range < mu - 2*sig)) |
          ((x_range > mu + 2*sig) & (x_range <= mu + 3*sig)),
    alpha=0.25, color=CORAL, label='±3σ  (99.7%)')
axes[0].set_title(f'3-Sigma Rule on Glucose\nμ={mu:.1f}, σ={sig:.1f}', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Glucose (mg/dL)')
axes[0].set_ylabel('Probability Density')
axes[0].legend()

# Right: Std Dev comparison across features (outlier impact)
features_var = ['Glucose', 'BloodPressure', 'Insulin', 'BMI', 'Age']
stds = [df[f].std() for f in features_var]
bars = axes[1].bar(features_var, stds, color=[TEAL, ORANGE, CORAL, NAVY, '#84B59F'], alpha=0.85, edgecolor='white')
axes[1].set_title('Standard Deviation by Feature\n(High SD = high variability)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Standard Deviation')
for bar, std in zip(bars, stds):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{std:.1f}',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print()
print('💡 Insulin has the highest std dev — it is very skewed with many zero values (missing data coded as 0).')

---
## 2. Probability Distributions — Modelling How Data Behaves

A **probability distribution** is a mathematical function that describes the possible values and their likelihoods for a random variable.

Why does this matter in ML?
> *"Real-world data is messy, incomplete, and noisy — so we model likelihoods instead of certainties."*

Distributions underpin:
- **Loss functions:** MSE assumes Gaussian errors; cross-entropy assumes Bernoulli
- **Model design:** Classification targets use Bernoulli; latent variables in deep learning use Gaussian priors
- **Generative AI:** GANs and VAEs sample from learned high-dimensional distributions

---

### 2.1 The Normal (Gaussian) Distribution — *The Bell Curve*

The **normal distribution** describes a continuous random variable whose values cluster around a central mean, with symmetric variability in both directions:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

It's ubiquitous because many natural phenomena — height, test scores, measurement errors — tend to follow this pattern. In ML, we frequently *assume* residuals (prediction errors) are normally distributed.

---

### 2.2 The Binomial Distribution — *Counting Successes*

The **binomial distribution** counts the number of successes in `n` independent trials, each with probability `p` of success:

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

This is the natural model for our diabetes dataset's `Outcome` column — each patient either *has* diabetes (1) or *doesn't* (0). The Bernoulli distribution is a special case where n=1.

In [ ]:
# ============================================================
# DISTRIBUTIONS — Normal & Binomial on the diabetes data
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Top Left: Normal distribution fit on Glucose ---
ax = axes[0, 0]
mu_g, sig_g = glucose.mean(), glucose.std()
ax.hist(glucose, bins=30, density=True, color=TEAL, alpha=0.6, edgecolor='white', label='Actual data')
x_range = np.linspace(glucose.min(), glucose.max(), 300)
ax.plot(x_range, stats.norm.pdf(x_range, mu_g, sig_g), color=CORAL, lw=2.5, label=f'Normal fit  μ={mu_g:.1f}, σ={sig_g:.1f}')
ax.set_title('Glucose — Normal Distribution Fit', fontweight='bold')
ax.set_xlabel('Glucose (mg/dL)')
ax.set_ylabel('Density')
ax.legend()

# --- Top Right: Check normality visually with Q-Q plot ---
ax = axes[0, 1]
stats.probplot(glucose, dist='norm', plot=ax)
ax.get_lines()[0].set(color=TEAL, alpha=0.6)
ax.get_lines()[1].set(color=CORAL, lw=2)
ax.set_title('Q-Q Plot: Is Glucose Normally Distributed?\n(Points on line = normal)', fontweight='bold')

# --- Bottom Left: Binomial — Diabetes Outcome distribution ---
ax = axes[1, 0]
n_trials = 30   # simulate drawing 30 patients
p_diabetes = df['Outcome'].mean()  # observed prevalence
k_values = np.arange(0, n_trials + 1)
binom_pmf = stats.binom.pmf(k_values, n_trials, p_diabetes)
ax.bar(k_values, binom_pmf, color=ORANGE, alpha=0.8, edgecolor='white')
ax.set_title(f'Binomial Distribution: Expected Diabetic Patients\nin a Random Sample of {n_trials} (p={p_diabetes:.2f})', fontweight='bold')
ax.set_xlabel('Number of Diabetic Patients')
ax.set_ylabel('Probability')
ax.axvline(n_trials * p_diabetes, color=CORAL, lw=2, linestyle='--', label=f'Expected mean = {n_trials*p_diabetes:.1f}')
ax.legend()

# --- Bottom Right: Actual Outcome proportions (Bernoulli) ---
ax = axes[1, 1]
outcome_counts = df['Outcome'].value_counts()
bars = ax.bar(['No Diabetes (0)', 'Diabetes (1)'],
              [outcome_counts[0]/len(df), outcome_counts[1]/len(df)],
              color=[TEAL, CORAL], alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, [outcome_counts[0]/len(df), outcome_counts[1]/len(df)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', fontsize=13, fontweight='bold')
ax.set_title('Bernoulli Distribution: Diabetes Outcomes\n(Each patient is one Bernoulli trial)', fontweight='bold')
ax.set_ylabel('Proportion')
ax.set_ylim(0, 0.85)

plt.tight_layout()
plt.show()

print(f'Diabetes prevalence in dataset: {p_diabetes:.1%}')
print(f'In a random group of 30 patients, we expect ~{n_trials*p_diabetes:.1f} to have diabetes.')

# Normality test
stat, p_value = stats.shapiro(glucose.sample(50, random_state=42))
print(f'\nShapiro-Wilk normality test on Glucose (n=50): p-value = {p_value:.4f}')
print('Interpretation:', 'Cannot reject normality (p > 0.05)' if p_value > 0.05 else 'Evidence against normality (p ≤ 0.05)')

---
## 3. Hypothesis Testing — *Making Decisions Under Uncertainty*

**Hypothesis testing** is a method that compares two opposite claims about a population and uses sample data to determine which is more likely to be true.

### The Framework

| Term | Meaning |
|------|---------|
| **Null hypothesis (H₀)** | The default assumption: *no effect, no relationship* |
| **Alternative hypothesis (Hₐ)** | What we're trying to show: *there IS an effect* |
| **p-value** | If H₀ is true, how surprising is our data? |
| **Significance level (α)** | Our threshold — typically 0.05 (5%) |

### Reading the p-value

> Think of the p-value as a **"Surprise Meter."**

- **Small p-value (< 0.05):** "This result is too weird to be random chance. **Reject H₀.**"
- **Large p-value (≥ 0.05):** "This could easily happen by chance. **Fail to reject H₀.**"

⚠️ **Common mistake:** A small p-value does NOT prove H₀ is false — it means the data is unlikely *if* H₀ were true.

### Types of Errors

| | H₀ is True | H₀ is False |
|-|------------|-------------|
| **Reject H₀** | ❌ Type I Error (false alarm) | ✅ Correct |
| **Don't Reject H₀** | ✅ Correct | ❌ Type II Error (missed detection) |

### Choosing the Right Test

| Test | When to Use | Example |
|------|-------------|----------|
| **t-Test** | Small sample, unknown variance, comparing 2 means | Glucose levels: diabetic vs non-diabetic |
| **z-Test** | Large sample (n > 30), known variance | Is avg BMI exactly 32? |
| **ANOVA** | Comparing 3+ group means | BMI across age groups |
| **Chi-Square** | Association between categorical variables | Diabetes outcome vs age group |

In [ ]:
# ============================================================
# HYPOTHESIS TESTING on the Diabetes Dataset
# ============================================================

# --- Test 1: t-Test ---
# H₀: Glucose levels are the same for diabetic and non-diabetic patients
# Hₐ: Glucose levels are significantly different

diabetic     = df[df['Outcome'] == 1]['Glucose']
non_diabetic = df[df['Outcome'] == 0]['Glucose']

t_stat, p_val = stats.ttest_ind(diabetic, non_diabetic)
print('=== t-Test: Glucose — Diabetic vs Non-Diabetic ===')
print(f'  H₀: Mean glucose is equal in both groups')
print(f'  t-statistic : {t_stat:.4f}')
print(f'  p-value     : {p_val:.2e}')
print(f'  Decision    : {"REJECT H₀ — Significant difference!" if p_val < 0.05 else "Fail to reject H₀"}')
print(f'  Diabetic mean:     {diabetic.mean():.1f} mg/dL')
print(f'  Non-diabetic mean: {non_diabetic.mean():.1f} mg/dL')

print()

# --- Test 2: ANOVA ---
# H₀: BMI is the same across age groups
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 30, 45, 100], labels=['Young (<30)', 'Middle (30-45)', 'Older (>45)'])
groups = [group['BMI'].dropna() for _, group in df.groupby('AgeGroup', observed=True)]
f_stat, p_anova = stats.f_oneway(*groups)
print('=== ANOVA: BMI across Age Groups ===')
print(f'  H₀: BMI is equal across age groups')
print(f'  F-statistic : {f_stat:.4f}')
print(f'  p-value     : {p_anova:.4f}')
print(f'  Decision    : {"REJECT H₀ — BMI differs across age groups" if p_anova < 0.05 else "Fail to reject H₀"}')

print()

# --- Test 3: Chi-Square ---
# H₀: Diabetes outcome is independent of age group
contingency = pd.crosstab(df['AgeGroup'], df['Outcome'])
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)
print('=== Chi-Square: Diabetes Outcome vs Age Group ===')
print(f'  H₀: Diabetes outcome is independent of age group')
print(f'  Chi-square: {chi2:.4f}')
print(f'  p-value   : {p_chi2:.4f}')
print(f'  Decision  : {"REJECT H₀ — Age group is associated with diabetes" if p_chi2 < 0.05 else "Fail to reject H₀"}')

In [ ]:
# ============================================================
# VISUALISE the Hypothesis Tests
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: t-Test distributions ---
ax = axes[0]
ax.hist(non_diabetic, bins=25, alpha=0.6, color=TEAL,  density=True, label=f'No Diabetes (n={len(non_diabetic)})')
ax.hist(diabetic,     bins=25, alpha=0.6, color=CORAL, density=True, label=f'Diabetes    (n={len(diabetic)})')
ax.axvline(non_diabetic.mean(), color=TEAL,  lw=2.5, linestyle='--')
ax.axvline(diabetic.mean(),     color=CORAL, lw=2.5, linestyle='--')
ax.set_title(f't-Test: Glucose Distribution\np={p_val:.2e} → Highly Significant', fontweight='bold')
ax.set_xlabel('Glucose (mg/dL)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

# --- Plot 2: ANOVA boxplots ---
ax = axes[1]
bmi_by_age = [df[df['AgeGroup'] == g]['BMI'].dropna() for g in ['Young (<30)', 'Middle (30-45)', 'Older (>45)']]
bp = ax.boxplot(bmi_by_age, patch_artist=True,
                boxprops=dict(facecolor=TEAL, alpha=0.6),
                medianprops=dict(color=CORAL, lw=2.5),
                whiskerprops=dict(color=NAVY),
                capprops=dict(color=NAVY))
ax.set_xticklabels(['Young\n(<30)', 'Middle\n(30-45)', 'Older\n(>45)'])
ax.set_title(f'ANOVA: BMI across Age Groups\np={p_anova:.4f}', fontweight='bold')
ax.set_ylabel('BMI')

# --- Plot 3: Chi-Square heatmap ---
ax = axes[2]
ct_norm = contingency.div(contingency.sum(axis=1), axis=0)
ct_norm.columns = ['No Diabetes', 'Diabetes']
sns.heatmap(ct_norm, annot=True, fmt='.1%', cmap='YlOrRd', ax=ax, linewidths=0.5, cbar=False)
ax.set_title(f'Chi-Square: Outcome % by Age Group\np={p_chi2:.4f}', fontweight='bold')
ax.set_ylabel('Age Group')

plt.tight_layout()
plt.show()

print()
print('💡 All three tests reject H₀:')
print('   • Glucose is significantly higher in diabetic patients')
print('   • BMI differs across age groups')
print('   • Age group is associated with diabetes risk')

---
## 4. Confidence Intervals — *Quantifying What We Don't Know*

### From Samples to Populations

In the real world, we almost never have access to an *entire* population — we work with samples. **Estimation** is the process of making inferences about unknown population parameters from sample data.

- **Point estimate:** A single best guess for a parameter (e.g., sample mean = 120.9 mg/dL)
- **Interval estimate:** A range that likely contains the true population value

### What is a Confidence Interval?

A **95% confidence interval (CI)** means:
> *"If we repeated this study 100 times, approximately 95 of those intervals would contain the true population parameter."*

For a mean with a large sample:

$$CI = \bar{x} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$

| Interval Width | Interpretation |
|----------------|----------------|
| **Wide interval** | Estimate is uncertain — could be almost anything |
| **Narrow interval** | Estimate is very reliable |

**What affects width?**
- Larger sample size → narrower CI (more data = more certainty)
- Higher variability → wider CI
- Higher confidence level (99% vs 95%) → wider CI

### Why CIs Matter in ML

When reporting model accuracy, a point estimate alone ("Our model is 85% accurate") is incomplete. A confidence interval gives stakeholders a realistic range and communicates uncertainty honestly.

In [ ]:
# ============================================================
# CONFIDENCE INTERVALS on the Diabetes Dataset
# ============================================================

def compute_ci(data, confidence=0.95):
    """Compute confidence interval for a mean."""
    n    = len(data)
    mean = np.mean(data)
    se   = stats.sem(data)  # standard error
    h    = se * stats.t.ppf((1 + confidence) / 2, df=n-1)
    return mean, mean - h, mean + h, h

# --- CIs for key features ---
features_ci = ['Glucose', 'BloodPressure', 'BMI', 'Age']
print('=== 95% Confidence Intervals for Feature Means ===')
ci_results = {}
for feat in features_ci:
    data = df[feat].replace(0, np.nan).dropna()
    mean, lo, hi, h = compute_ci(data)
    ci_results[feat] = (mean, lo, hi, h)
    print(f'  {feat:22s}: {mean:.2f}  95% CI [{lo:.2f}, {hi:.2f}]  (±{h:.2f})')

print()

# --- Effect of sample size on CI width ---
print('=== Effect of Sample Size on CI Width (Glucose) ===')
for n in [10, 30, 100, 300, 768]:
    sample = glucose.sample(min(n, len(glucose)), random_state=42)
    _, lo, hi, h = compute_ci(sample)
    print(f'  n={n:4d}: CI width = {2*h:.2f}  [{lo:.1f}, {hi:.1f}]')

print()
print('💡 As sample size grows, the CI narrows — more data = more certainty.')

In [ ]:
# ============================================================
# VISUALISE Confidence Intervals
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: CI forest plot by feature ---
ax = axes[0]
y_pos = range(len(features_ci))
means = [ci_results[f][0] for f in features_ci]
errors = [ci_results[f][3] for f in features_ci]

ax.barh(y_pos, means, xerr=errors, align='center',
        color=TEAL, alpha=0.7, ecolor=CORAL, capsize=8, height=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(features_ci, fontsize=11)
ax.set_title('95% Confidence Intervals for Feature Means\n(Error bars show CI range)', fontweight='bold')
ax.set_xlabel('Value')
for i, (mean, lo, hi, h) in enumerate(ci_results.values()):
    ax.text(mean + h + 0.5, i, f'[{lo:.1f}, {hi:.1f}]', va='center', fontsize=9, color=NAVY)

# --- Plot 2: CI width vs sample size ---
ax = axes[1]
sample_sizes = [10, 20, 30, 50, 75, 100, 150, 200, 300, 500, 768]
ci_widths = []
for n in sample_sizes:
    sample = glucose.sample(min(n, len(glucose)), random_state=42)
    _, _, _, h = compute_ci(sample)
    ci_widths.append(2 * h)

ax.plot(sample_sizes, ci_widths, color=CORAL, lw=2.5, marker='o', markersize=6)
ax.fill_between(sample_sizes, ci_widths, alpha=0.15, color=CORAL)
ax.set_title('CI Width Shrinks as Sample Size Grows\n(Glucose)', fontweight='bold')
ax.set_xlabel('Sample Size (n)')
ax.set_ylabel('95% CI Width (mg/dL)')
ax.axhline(min(ci_widths), color=NAVY, lw=1, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 5. Correlation vs. Causation — *The Most Important Distinction in Data Science*

### Covariance

**Covariance** measures whether two variables move together:

$$\text{Cov}(X, Y) = \frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})$$

- **Positive covariance:** Both increase together
- **Negative covariance:** One increases as the other decreases
- **Problem:** The scale depends on units — "2000 kg·mg/dL" is hard to interpret

### Correlation

**Pearson correlation** is the *standardised* version of covariance, always between -1 and +1:

$$r = \frac{\text{Cov}(X, Y)}{\sigma_X \cdot \sigma_Y}$$

| r value | Strength | Direction |
|---------|----------|----------|
| +0.7 to +1.0 | Strong | Positive |
| +0.3 to +0.7 | Moderate | Positive |
| -0.3 to +0.3 | Weak | — |
| -0.7 to -0.3 | Moderate | Negative |
| -1.0 to -0.7 | Strong | Negative |

---

### ⚠️ Correlation ≠ Causation

This is the most repeated lesson in statistics — and the most often violated in practice.

> *"Models learn correlations, not causes. A feature can improve predictions without being the true driver."*

**Classic examples of spurious correlation:**
- Ice cream sales and drowning rates both rise in summer (confounded by *heat*)
- Nicolas Cage movies correlate with pool drowning deaths (pure coincidence)
- In our dataset: age correlates with diabetes, but age *causes* diabetes?

**Risk in ML:** Using correlated but non-causal features means models can break catastrophically when the data distribution changes (e.g., deploying to a different population).

In [ ]:
# ============================================================
# CORRELATION ANALYSIS on the Diabetes Dataset
# ============================================================

# --- Correlation matrix ---
corr_matrix = df.drop('AgeGroup', axis=1).corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
ax = axes[0]
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, mask=mask, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix\n(Lower triangle only)', fontweight='bold')

# Correlation with Outcome specifically
ax = axes[1]
outcome_corr = corr_matrix['Outcome'].drop('Outcome').sort_values()
colors = [CORAL if x > 0 else TEAL for x in outcome_corr]
bars = ax.barh(outcome_corr.index, outcome_corr.values, color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_title('Correlation with Diabetes Outcome\n(Positive = increases risk)', fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient (r)')
for bar, val in zip(bars, outcome_corr.values):
    x_pos = val + 0.005 if val >= 0 else val - 0.005
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n=== Top Correlates with Diabetes Outcome ===')
top = outcome_corr.abs().sort_values(ascending=False)
for feat, val in top.items():
    direction = '↑' if outcome_corr[feat] > 0 else '↓'
    print(f'  {direction} {feat:30s}: |r| = {val:.3f}')

In [ ]:
# ============================================================
# SCATTER PLOTS — visualise specific correlations
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pairs = [
    ('Glucose', 'BMI', 'Moderate correlation'),
    ('Age', 'Pregnancies', 'Expected positive trend'),
    ('BloodPressure', 'DiabetesPedigreeFunction', 'Weak / near-zero correlation'),
]

for ax, (x_feat, y_feat, note) in zip(axes, pairs):
    diabetic_mask = df['Outcome'] == 1
    r, p = stats.pearsonr(df[x_feat].dropna(), df[y_feat].dropna())

    ax.scatter(df.loc[~diabetic_mask, x_feat], df.loc[~diabetic_mask, y_feat],
               color=TEAL, alpha=0.4, s=25, label='No Diabetes')
    ax.scatter(df.loc[diabetic_mask, x_feat], df.loc[diabetic_mask, y_feat],
               color=CORAL, alpha=0.4, s=25, label='Diabetes')

    # Regression line
    m, b = np.polyfit(df[x_feat], df[y_feat], 1)
    x_line = np.linspace(df[x_feat].min(), df[x_feat].max(), 100)
    ax.plot(x_line, m * x_line + b, color=NAVY, lw=2, linestyle='--')

    ax.set_title(f'{x_feat} vs {y_feat}\nr = {r:.3f}  ({note})', fontweight='bold', fontsize=10)
    ax.set_xlabel(x_feat)
    ax.set_ylabel(y_feat)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print()
print('⚠️  REMINDER: Even r=0.47 (Glucose vs BMI) does NOT mean BMI causes diabetes.')
print('   Both may be driven by lifestyle factors, genetics, or other confounders.')

---
## 6. Introduction to Regression — *Predicting with Statistics*

### 6.1 Linear Regression

**Linear regression** models the relationship between a continuous output `y` and one or more input features `X`:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \epsilon$$

Where:
- $\beta_0$ = intercept (value of y when all x = 0)
- $\beta_1, \beta_2, \ldots$ = coefficients (how much y changes per unit of x)
- $\epsilon$ = error term (what the model can't explain)

The model finds the line (or hyperplane) that **minimises the sum of squared residuals** (SSR):

$$SSR = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

This is directly related to our descriptive statistics — we're essentially minimising variance around our predictions.

### 6.2 Logistic Regression — *When the Outcome is Binary*

For our diabetes dataset, the target is 0 or 1 — not a continuous value. **Logistic regression** extends linear regression by passing the output through a sigmoid function:

$$P(y=1 | X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \ldots)}}$$

This ensures predictions are always between 0 and 1 — interpretable as probabilities. This connects back to our probability distributions section: the output is a Bernoulli probability.

### Key Evaluation Metrics

| Metric | Formula | Meaning |
|--------|---------|----------|
| **R²** | $1 - SS_{res}/SS_{tot}$ | How much variance the model explains (0 to 1) |
| **MSE** | $\frac{1}{n}\sum(y - \hat{y})^2$ | Average squared error |
| **RMSE** | $\sqrt{MSE}$ | Error in original units |

In [ ]:
# ============================================================
# LINEAR REGRESSION: Predict BMI from Glucose + Age
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Clean data
df_clean = df.replace(0, np.nan).dropna(subset=['BMI', 'Glucose', 'Age', 'BloodPressure'])

X_lin = df_clean[['Glucose', 'Age', 'BloodPressure']]
y_lin = df_clean['BMI']

X_train, X_test, y_train, y_test = train_test_split(X_lin, y_lin, test_size=0.2, random_state=42)

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test)

r2  = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print('=== Linear Regression: Predicting BMI ===')
print(f'  R²   = {r2:.4f}  (model explains {r2*100:.1f}% of BMI variance)')
print(f'  RMSE = {rmse:.4f} kg/m²')
print()
print('  Feature Coefficients:')
for feat, coef in zip(X_lin.columns, lin_model.coef_):
    print(f'    {feat:25s}: {coef:+.4f}')
print(f'    {"Intercept":25s}: {lin_model.intercept_:+.4f}')

In [ ]:
# ============================================================
# LOGISTIC REGRESSION: Predict Diabetes Outcome
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

X_log = df_clean[['Glucose', 'BMI', 'Age', 'BloodPressure', 'Pregnancies', 'DiabetesPedigreeFunction']]
y_log = df_clean['Outcome']

X_tr, X_te, y_tr, y_te = train_test_split(X_log, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_tr_s, y_tr)
y_pred_log = log_model.predict(X_te_s)
y_prob_log = log_model.predict_proba(X_te_s)[:, 1]

print('=== Logistic Regression: Predicting Diabetes ===')
print(classification_report(y_te, y_pred_log, target_names=['No Diabetes', 'Diabetes']))

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion matrix
cm = confusion_matrix(y_te, y_pred_log)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0],
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'],
            linewidths=0.5)
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature coefficients — which features drive the prediction?
coefs = pd.Series(log_model.coef_[0], index=X_log.columns).sort_values()
colors_c = [CORAL if c > 0 else TEAL for c in coefs]
axes[1].barh(coefs.index, coefs.values, color=colors_c, alpha=0.85, edgecolor='white')
axes[1].axvline(0, color='black', lw=1)
axes[1].set_title('Logistic Regression Coefficients\n(Standardised — larger = more impact)', fontweight='bold')
axes[1].set_xlabel('Coefficient')

# Predicted probabilities
axes[2].hist(y_prob_log[y_te == 0], bins=20, alpha=0.65, color=TEAL, label='No Diabetes', density=True)
axes[2].hist(y_prob_log[y_te == 1], bins=20, alpha=0.65, color=CORAL, label='Diabetes', density=True)
axes[2].axvline(0.5, color=NAVY, lw=2, linestyle='--', label='Decision threshold = 0.5')
axes[2].set_title('Predicted Probability Distribution\n(Model output = Bernoulli probability)', fontweight='bold')
axes[2].set_xlabel('P(Diabetes)')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Linear regression residuals
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
residuals = y_test - y_pred
axes[0].scatter(y_pred, residuals, color=TEAL, alpha=0.5, s=30)
axes[0].axhline(0, color=CORAL, lw=2, linestyle='--')
axes[0].set_title('Residual Plot: Linear Regression (BMI)\n(Should be random around 0)', fontweight='bold')
axes[0].set_xlabel('Predicted BMI')
axes[0].set_ylabel('Residual')

axes[1].scatter(y_test, y_pred, color=ORANGE, alpha=0.5, s=30)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[1].plot([min_val, max_val], [min_val, max_val], color=NAVY, lw=2, linestyle='--', label='Perfect prediction')
axes[1].set_title(f'Actual vs Predicted BMI\nR² = {r2:.3f}', fontweight='bold')
axes[1].set_xlabel('Actual BMI')
axes[1].set_ylabel('Predicted BMI')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. Interactive Session: Comprehensive Statistical Analysis

Now we bring everything together in a full exploratory analysis of the diabetes dataset — the kind of workflow you'd actually run at the start of any real data science project.

### The Workflow
1. **Profile the data** — descriptive stats, missing values, distributions
2. **Test hypotheses** — are feature differences between groups real or random?
3. **Measure relationships** — correlations, regression
4. **Communicate uncertainty** — confidence intervals on key estimates

In [ ]:
# ============================================================
# COMPREHENSIVE EDA REPORT
# ============================================================

print('=' * 60)
print('       DIABETES DATASET — STATISTICAL REPORT')
print('=' * 60)

# --- Data Quality ---
print('\n📋 DATA QUALITY')
print(f'  Records : {len(df)}')
print(f'  Features: {df.shape[1] - 1}')
print(f'  Target  : Outcome (1=Diabetic, 0=Not Diabetic)')

# Zero values in medical features indicate missing data
zero_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('\n  Zero values (likely missing):')
for feat in zero_features:
    zeros = (df[feat] == 0).sum()
    pct = zeros / len(df) * 100
    print(f'    {feat:25s}: {zeros:3d} zeros ({pct:.1f}%)')

# --- Summary Statistics by Outcome ---
print('\n📊 MEAN ± STD BY OUTCOME GROUP')
numeric_feats = ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'Age', 'DiabetesPedigreeFunction']
summary = df.groupby('Outcome')[numeric_feats].agg(['mean', 'std']).round(2)
for feat in numeric_feats:
    m0 = df[df['Outcome']==0][feat].mean()
    s0 = df[df['Outcome']==0][feat].std()
    m1 = df[df['Outcome']==1][feat].mean()
    s1 = df[df['Outcome']==1][feat].std()
    t, p = stats.ttest_ind(df[df['Outcome']==1][feat], df[df['Outcome']==0][feat])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {feat:30s}: No Diab={m0:.1f}±{s0:.1f}  Diab={m1:.1f}±{s1:.1f}  {sig}')

print('\n  Significance: *** p<0.001  ** p<0.01  * p<0.05  ns=not significant')

In [ ]:
# ============================================================
# FULL DISTRIBUTION PANEL
# ============================================================

features_plot = ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'Age', 'DiabetesPedigreeFunction']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Distributions by Diabetes Outcome', fontsize=16, fontweight='bold', y=1.01)

for ax, feat in zip(axes.flatten(), features_plot):
    d0 = df[df['Outcome'] == 0][feat].replace(0, np.nan).dropna()
    d1 = df[df['Outcome'] == 1][feat].replace(0, np.nan).dropna()

    ax.hist(d0, bins=20, alpha=0.55, color=TEAL,  density=True, label=f'No Diabetes (n={len(d0)})')
    ax.hist(d1, bins=20, alpha=0.55, color=CORAL, density=True, label=f'Diabetes    (n={len(d1)})')

    # Means
    ax.axvline(d0.mean(), color=TEAL,  lw=2, linestyle='--', alpha=0.9)
    ax.axvline(d1.mean(), color=CORAL, lw=2, linestyle='--', alpha=0.9)

    # Test
    t, p = stats.ttest_ind(d1, d0)
    sig_label = f'p={p:.4f}' if p >= 0.0001 else f'p={p:.2e}'

    ax.set_title(f'{feat}\n({sig_label})', fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PRACTICE: Apply what you've learned!
# ============================================================

print('=' * 60)
print('  PRACTICE EXERCISES — Try these yourself!')
print('=' * 60)
print()
print('1. DESCRIPTIVE STATISTICS')
print('   → Compute mean, median, and std of the Insulin column.')
print('   → Why is the mean so much larger than the median?')
print()
print('2. DISTRIBUTIONS')
print('   → Plot the distribution of Pregnancies.')
print('   → Is it normal, left-skewed, or right-skewed?')
print()
print('3. HYPOTHESIS TESTING')
print('   → Test if DiabetesPedigreeFunction differs between groups.')
print('   → What is your H₀? What does a p<0.05 tell you?')
print()
print('4. CONFIDENCE INTERVALS')
print('   → Compute 90%, 95%, and 99% CIs for mean Glucose.')
print('   → How does the width change across confidence levels?')
print()
print('5. CORRELATION')
print('   → Find the feature most correlated with Glucose.')
print('   → Does a high correlation mean it causes Glucose to rise?')
print()
print('6. REGRESSION')
print('   → Add Insulin as a feature to the logistic regression.')
print('   → Does model accuracy improve?')
print()
print('--- STARTER CODE ---')
print()

# Starter for exercise 1
insulin_clean = df['Insulin'].replace(0, np.nan).dropna()
print(f'Exercise 1 starter:')
print(f'  Insulin — Mean: {insulin_clean.mean():.2f}, Median: {insulin_clean.median():.2f}, Std: {insulin_clean.std():.2f}')
print(f'  Mean >> Median → strong right skew (a few very high insulin values pull the mean up)')

# Starter for exercise 3
dpf0 = df[df['Outcome'] == 0]['DiabetesPedigreeFunction']
dpf1 = df[df['Outcome'] == 1]['DiabetesPedigreeFunction']
t, p = stats.ttest_ind(dpf0, dpf1)
print(f'\nExercise 3 starter:')
print(f'  H₀: DiabetesPedigreeFunction is equal in both groups')
print(f'  t={t:.4f}, p={p:.4f} → {"Reject H₀" if p < 0.05 else "Fail to reject H₀"}')

---
## ✅ Summary: What We've Learned

| Concept | Key Takeaway | Applied to Diabetes Dataset |
|---------|-------------|-----------------------------|
| **Mean/Median/Mode** | Use mean for symmetric data; median for skewed; mode for categorical | Glucose is roughly symmetric (mean ≈ median) |
| **Variance/Std Dev** | Measures spread; outliers inflate both | Insulin is highly variable with many zeros |
| **Normal Distribution** | Bell curve; models many natural phenomena and ML error terms | Glucose roughly normal; Q-Q plot confirms |
| **Binomial Distribution** | Counts successes in n trials | Diabetes outcomes modelled as Bernoulli trials |
| **Hypothesis Testing** | Reject H₀ only when data is sufficiently surprising (p < 0.05) | Glucose is significantly higher in diabetic patients |
| **Confidence Intervals** | A range for uncertainty; wider = less certain | 95% CI for mean Glucose is tight (large sample) |
| **Correlation** | Standardised covariance, always -1 to +1 | Glucose has highest correlation with Outcome |
| **Correlation ≠ Causation** | ML learns patterns, not causes | High glucose correlates with diabetes but doesn't *cause* it in isolation |
| **Linear Regression** | Minimises squared residuals to fit a line | Predicts BMI from Glucose, Age, Blood Pressure |
| **Logistic Regression** | Maps linear output to probability via sigmoid | Predicts P(Diabetes) for each patient |

---

### Next Steps

- **Feature Engineering:** Use your understanding of distributions to transform skewed features (log transform Insulin)
- **Advanced Testing:** Explore non-parametric tests (Mann-Whitney U) for heavily skewed data
- **Bayesian Statistics:** Update beliefs about diabetes risk as new patient data arrives
- **Model Evaluation:** Compute confidence intervals around model accuracy metrics

---
*Built on the Pima Indians Diabetes Dataset | AI4SID Foundations of Statistics Workshop*